## 1. Importar Bibliotecas e Verificar Ambiente

Validar que todas as dependências estão instaladas e o ambiente está corretamente configurado.

In [ ]:
# Importar bibliotecas essenciais
import sys
print(f"Python version: {sys.version}")

# Verificar Ultralytics
try:
    from ultralytics import YOLO
    import ultralytics
    print(f"✅ Ultralytics: {ultralytics.__version__}")
except ImportError as e:
    print(f"❌ Erro ao importar Ultralytics: {e}")

# Importar bibliotecas de análise
import os
import json
import yaml
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

print("✅ Todas as bibliotecas importadas com sucesso!")

# Configurar matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
np.set_printoptions(precision=3, suppress=True)

## 2. Inferência Local com Ultralytics YOLO

Carregar um modelo pré-treinado YOLOv8 e executar inferência em uma imagem de exemplo. 
O modelo `yolov8n.pt` é a versão "nano" (mais rápida e menor).

In [ ]:
# Carregar o modelo pré-treinado
print("Carregando modelo YOLOv8n...")
model = YOLO("yolov8n.pt")
print(f"✅ Modelo carregado com sucesso!")
print(f"   - Task: {model.task}")
print(f"   - Classes: {len(model.names)}")
print(f"   - Nomes das classes (primeiras 10): {list(model.names.values())[:10]}")

In [ ]:
# Teste 1: Inferência com imagem local (se existir)
img_path = "/workspaces/yolo-pytorch-studies/bus.jpg"
if os.path.exists(img_path):
    print(f"\n📌 Teste 1: Inferência em imagem local")
    print(f"   Arquivo: {img_path}")
    
    # Executar inferência
    results = model(img_path, verbose=False)
    result = results[0]
    
    print(f"   ✅ Inferência concluída!")
    print(f"   - Objetos detectados: {len(result.boxes)}")
    
    if len(result.boxes) > 0:
        # Exibir detecções
        print(f"   - Detalhes das detecções:")
        boxes = result.boxes
        for i, (box, conf, cls) in enumerate(zip(boxes.xyxy, boxes.conf, boxes.cls)):
            class_name = model.names[int(cls)]
            print(f"     [{i}] {class_name} - Confiança: {conf:.3f}")
        
        # Plot
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
        im = Image.open(img_path)
        ax.imshow(im)
        
        # Desenhar boxes
        for box, conf, cls in zip(boxes.xyxy, boxes.conf, boxes.cls):
            x1, y1, x2, y2 = box.cpu().numpy()
            class_name = model.names[int(cls)]
            ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='red', linewidth=2))
            ax.text(x1, y1-10, f"{class_name} {conf:.2f}", color='red', fontsize=10)
        
        ax.set_title("YOLOv8 Detections")
        ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print(f"⚠️  Imagem local não encontrada em {img_path}")

print("\n📌 Teste 2: Inferência com URL remota")
try:
    print("   Carregando imagem de: https://ultralytics.com/images/bus.jpg")
    remote_results = model("https://ultralytics.com/images/bus.jpg", verbose=False)
    remote_result = remote_results[0]
    print(f"   ✅ Inferência remota concluída!")
    print(f"   - Objetos detectados: {len(remote_result.boxes)}")
except Exception as e:
    print(f"   ⚠️  Erro na inferência remota (pode ser problema de rede): {e}")

## 3. Treinamento Curto com COCO128

Executar um ciclo de treinamento rápido (3 épocas) com o dataset COCO128 para validar o fluxo de treinamento.
Este é um dataset pequeno (128 imagens) usado apenas para prototipagem rápida.

In [ ]:
# Verificar workspace structure
print("📂 Estrutura do dataset COCO128:")
coco_path = Path("/workspaces/yolo-pytorch-studies/datasets/coco128")
print(f"   Imagens: {len(list((coco_path / 'images' / 'train2017').glob('*.jpg')))} arquivos")
print(f"   Labels: {len(list((coco_path / 'labels' / 'train2017').glob('*.txt')))} arquivos")

# Informações sobre o dataset
print("\n📊 Informações do dataset COCO128:")
print("   - Total de imagens: 128")
print("   - Classes: 80 (COCO dataset - objetos comuns)")
print("   - Resolução típica: variável (640x480 a 640x640)")
print("   - Objetivo: Detecção de objetos (modo 'detect')")

print("\n⏳ Comentário: O treinamento será realizado usando os scripts no workspace")
print("   Script: treinar_coco128.py")
print("   Para executar manualmente: /workspaces/yolo-pytorch-studies/.venv/bin/python treinar_coco128.py")
print("\n   Nota: Treinamento com 3 épocas dura ~5-10 minutos em GPU, mais em CPU.")

## 4. Exportar Modelo para Engine TensorRT

Demonstrar como exportar modelos YOLO para diferentes formatos, incluindo TensorRT (engine NVIDIA).
Este é um passo essencial para deployar no DeepStream.

In [ ]:
# Demonstrar formatos de exportação disponíveis
print("📦 Formatos de exportação disponíveis no Ultralytics:\n")

export_formats = {
    "pytorch": "Modelo PyTorch nativo (.pt)",
    "torchscript": "TorchScript (.torchscript)",
    "onnx": "ONNX - Open Neural Network Exchange (.onnx)",
    "openvino": "OpenVINO - Intel format (.xml/.bin)",
    "tflite": "TensorFlow Lite - Mobile/Edge (.tflite)",
    "pb": "TensorFlow SavedModel (.pb)",
    "saved_model": "TensorFlow SavedModel (dir/)",
    "engine": "TensorRT - NVIDIA GPU (.engine)",
    "coreml": "Core ML - Apple (.mlmodel)",
    "model": "Keras model (.h5)",
}

for fmt, desc in export_formats.items():
    print(f"  • {fmt:15} -> {desc}")

print("\n🎯 Para pipeline DeepStream, usar:")
print("   model.export(format='engine')")
print("   ou")
print("   model.export(format='onnx')  # Se não tiver CUDA disponível")

print("\n⚙️  Fluxo de exportação:")
print("   1. Modelo YOLO treinado (.pt)")
print("   2. Exportar para ONNX (.onnx)")
print("   3. Converter ONNX → TensorRT (.engine) usando trtexec")
print("   4. Carregar engine no DeepStream via plugin nvinfer")

print("\n💡 Exemplo de código (não executado neste notebook):")
print("""
from ultralytics import YOLO

model = YOLO('best.pt')  # Modelo treinado

# Exportar para ONNX
export_path = model.export(format='onnx', imgsz=640)
print(f"ONNX exportado para: {export_path}")

# Exportar para TensorRT (requer CUDA + TensorRT instalado)
export_path = model.export(format='engine', imgsz=640, device=0)
print(f"Engine TensorRT exportado para: {export_path}")
""")

## 5. Analisar Dataset de Veículos

Exploração dos dados dos veículos (cor, modelo, tipo). Verificar volume por classe, distribuição e identificar classes desbalanceadas.

In [ ]:
# Função para analisar dataset YOLO
def analyze_yolo_dataset(dataset_path, dataset_name="Dataset"):
    """Analisa estrutura, classes e distribuição de um dataset YOLO"""
    
    dataset_path = Path(dataset_path)
    
    if not dataset_path.exists():
        print(f"⚠️  {dataset_name} não encontrado em: {dataset_path}")
        return None
    
    print(f"\n{'='*70}")
    print(f"ANÁLISE: {dataset_name}")
    print(f"Path: {dataset_path}")
    print(f"{'='*70}\n")
    
    # Procurar arquivos
    images_dir = dataset_path / "images"
    labels_dir = dataset_path / "labels"
    
    if not images_dir.exists() or not labels_dir.exists():
        print(f"❌ Estrutura esperada não encontrada (faltam: images/ ou labels/)")
        return None
    
    # Coletar estatísticas
    stats = {
        'total_images': 0,
        'total_objects': 0,
        'class_distribution': Counter(),
        'resolutions': [],
        'split_stats': {},
        'bbox_sizes': {'width': [], 'height': []},
    }
    
    # Processar splits (train, val, test)
    for split_dir in images_dir.glob('*'):
        if not split_dir.is_dir():
            continue
        
        split_name = split_dir.name
        images = list(split_dir.glob('*.jpg')) + list(split_dir.glob('*.png'))
        num_images = len(images)
        num_objects = 0
        
        stats['total_images'] += num_images
        stats['split_stats'][split_name] = num_images
        
        # Processar labels
        for img_path in images:
            try:
                img = Image.open(img_path)
                stats['resolutions'].append(img.size)
            except:
                pass
            
            label_path = labels_dir / split_name / img_path.stem
            if label_path.with_suffix('.txt').exists():
                with open(label_path.with_suffix('.txt'), 'r') as f:
                    lines = f.readlines()
                    for line in lines:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            class_id = int(parts[0])
                            stats['class_distribution'][class_id] += 1
                            stats['bbox_sizes']['width'].append(float(parts[3]))
                            stats['bbox_sizes']['height'].append(float(parts[4]))
                            num_objects += 1
        
        stats['total_objects'] += num_objects
    
    # Exibir resultados
    print(f"📊 RESUMO")
    print(f"  • Total de imagens: {stats['total_images']}")
    print(f"  • Total de objetos: {stats['total_objects']}")
    if stats['total_images'] > 0:
        print(f"  • Objetos/imagem (média): {stats['total_objects'] / stats['total_images']:.2f}")
    
    # Por split
    if stats['split_stats']:
        print(f"\n📂 Distribuição por split:")
        for split, count in stats['split_stats'].items():
            pct = (count / stats['total_images'] * 100) if stats['total_images'] > 0 else 0
            print(f"  • {split:12} {count:4} imagens ({pct:5.1f}%)")
    
    # Classes
    if stats['class_distribution']:
        print(f"\n🏷️  Distribuição de classes:")
        total = sum(stats['class_distribution'].values())
        for class_id in sorted(stats['class_distribution'].keys()):
            count = stats['class_distribution'][class_id]
            pct = (count / total * 100) if total > 0 else 0
            bar = '█' * int(pct / 2)
            print(f"  • Classe {class_id:2d}: {count:6d} ({pct:5.1f}%) {bar}")
    
    # Resoluções
    if stats['resolutions']:
        resolutions = np.array(stats['resolutions'])
        print(f"\n📐 Resoluções (WxH):")
        print(f"  • Min: {resolutions.min(axis=0)}")
        print(f"  • Max: {resolutions.max(axis=0)}")
        print(f"  • Média: {resolutions.mean(axis=0).astype(int)}")
    
    return stats

# Analisar dataset do COCO128
coco_stats = analyze_yolo_dataset(
    "/workspaces/yolo-pytorch-studies/datasets/coco128", 
    "COCO128 (Detecção de Objetos)"
)

# Verificar se dataset de veículos existe
veiculos_path = "/workspaces/datasets/veiculos"
if Path(veiculos_path).exists():
    print("\n")
    veiculos_stats = analyze_yolo_dataset(veiculos_path, "Dataset de Veículos")
else:
    print(f"\n⚠️  Dataset de veículos não encontrado em: {veiculos_path}")
    print("   Nota: O arquivo config referencia '/workspaces/datasets/veiculos'")
    print("   Este dataset ainda precisa ser criado/copiado")

## 6. Analisar Dataset de Atributos Faciais

Exploração dos atributos faciais (barba, óculos, expressão, etc.). 
Verificar balanceamento de classes e identificar possíveis ambiguidades.

In [ ]:
# Informações sobre dataset de atributos faciais
facial_path = "/workspaces/datasets/facial_attributes"

print("\n" + "="*70)
print("ANÁLISE: Dataset de Atributos Faciais")
print("="*70 + "\n")

if Path(facial_path).exists():
    facial_stats = analyze_yolo_dataset(facial_path, "Dataset de Atributos Faciais")
else:
    print(f"⚠️  Dataset não encontrado em: {facial_path}\n")
    
    print("📋 SCHEMA DE CLASSES SUGERIDO PARA ATRIBUTOS FACIAIS:\n")
    
    # Taxonomia sugerida
    schema = {
        "Barba": {
            "classes": [
                "sem_barba",
                "barba_curta", 
                "barba_longa",
                "cavanhaque",
                "bigode"
            ],
            "notas": "Evitar ambiguidade: definir limites de tamanho"
        },
        "Óculos": {
            "classes": [
                "sem_oculos",
                "oculos_escuros",
                "oculos_transparente",
                "oculos_armacao"
            ],
            "notas": "Clareza: óculos pode ser escuro total, com transparência, etc."
        },
        "Cabelo": {
            "classes": [
                "cabelo_curto",
                "cabelo_medio",
                "cabelo_longo",
                "careca"
            ],
            "notas": "Usar referências visuais para definir comprimento"
        },
        "Expressão": {
            "classes": [
                "neutral",
                "sorriso",
                "serio",
                "surpreso",
                "triste"
            ],
            "notas": "Validar consistência entre anotadores"
        },
        "Gênero": {
            "classes": [
                "feminino",
                "masculino"
            ],
            "notas": "Se necessário, considerar classe 'ambiguo'"
        }
    }
    
    for attribute, info in schema.items():
        print(f"🏷️  {attribute}:")
        for cls in info['classes']:
            print(f"     • {cls}")
        print(f"     ℹ️  {info['notas']}\n")

print("⚠️  PROBLEMAS COMUNS EM DATASETS FACIAIS:\n")
issues = [
    "✗ Desbalanceamento extremo: 'sem_barba' >> outros (80%+ das amostras)",
    "✗ Ambiguidade: definições vagas entre classes (ex: barba curta vs média)",
    "✗ Múltiplos atributos: mesma pessoa/face com múltiplos labels",
    "✗ Baixa qualidade de anotação: inconsistência entre anotadores",
    "✗ Viés demográfico: predominância de um gênero/etnia"
]
for issue in issues:
    print(f"  {issue}")

print("\n💡 RECOMENDAÇÕES:")
recomendations = [
    "1. Criar guidelines visuais com exemplos para cada classe",
    "2. Validar inter-anotador (Cohen's kappa ou similar)",
    "3. Usar técnicas de balanceamento: data augmentation, oversampling",
    "4. Considerar multi-label se um rosto pode ter múltiplos atributos",
    "5. Revisar periodicamente: descartar anotações inconsistentes"
]
for rec in recomendations:
    print(f"  {rec}")

## 7. Registrar Experimentos Manualmente

Setup inicial para rastreamento de experimentos. Duas opções: registro manual simples ou MLflow.

In [ ]:
# Sistema simples de registro de experimentos
import json
from datetime import datetime

class ExperimentTracker:
    """Rastreador simples de experimentos"""
    
    def __init__(self, log_file="experiments_log.json"):
        self.log_file = log_file
        self.experiments = self._load_experiments()
    
    def _load_experiments(self):
        """Carrega experimentos anteriores"""
        if Path(self.log_file).exists():
            with open(self.log_file, 'r') as f:
                return json.load(f)
        return []
    
    def _save_experiments(self):
        """Salva experimentos em arquivo"""
        with open(self.log_file, 'w') as f:
            json.dump(self.experiments, f, indent=2, default=str)
    
    def log_experiment(self, name, dataset, config, results=None, notes=""):
        """Registra um novo experimento"""
        
        experiment = {
            "timestamp": datetime.now().isoformat(),
            "name": name,
            "dataset": dataset,
            "config": config,
            "results": results,
            "notes": notes
        }
        
        self.experiments.append(experiment)
        self._save_experiments()
        
        print(f"✅ Experimento registrado: {name}")
        return experiment
    
    def list_experiments(self):
        """Lista todos os experimentos"""
        print(f"\n📋 Total de experimentos: {len(self.experiments)}\n")
        
        for i, exp in enumerate(self.experiments, 1):
            print(f"{i}. {exp['name']}")
            print(f"   Dataset: {exp['dataset']}")
            print(f"   Data: {exp['timestamp']}")
            if exp['notes']:
                print(f"   Notas: {exp['notes']}")
            print()

# Criar rastreador
tracker = ExperimentTracker("/workspaces/yolo-pytorch-studies/experiments_log.json")

# Exemplo de registro
print("📝 EXEMPLO: Registrando um experimento\n")

sample_config = {
    "model": "yolov8n.pt",
    "dataset": "coco128",
    "epochs": 3,
    "imgsz": 640,
    "batch": 16,
    "device": "cpu"
}

tracker.log_experiment(
    name="Baseline COCO128",
    dataset="COCO128",
    config=sample_config,
    results={
        "mAP50": 0.456,
        "mAP50-95": 0.298
    },
    notes="Treinamento rápido para validar pipeline"
)

print("\n📊 Experimentos registrados:")
tracker.list_experiments()

print("\n" + "="*70)
print("OPÇÃO: Se preferir MLflow para rastreamento mais robusto:\n")

mlflow_info = """
1. Instalar: pip install mlflow

2. Iniciar servidor: mlflow ui --host 0.0.0.0 --port 5000

3. Usar no notebook:
   
   import mlflow
   mlflow.set_tracking_uri("http://localhost:5000")
   mlflow.set_experiment("meu-experimento")
   
   with mlflow.start_run():
       mlflow.log_param("epochs", 50)
       mlflow.log_param("batch_size", 16)
       mlflow.log_metric("mAP50", 0.456)
       mlflow.log_artifact("model.pt")

4. Acessar dashboard em: http://localhost:5000
"""

print(mlflow_info)

print("\n💡 VANTAGENS DE CADA ABORDAGEM:\n")

comparison = pd.DataFrame({
    "Feature": [
        "Setup",
        "Complexidade",
        "Persistência",
        "Visualização",
        "Colaboração",
        "Recomendado para"
    ],
    "JSON Manual": [
        "Nenhum",
        "Baixa",
        "Arquivo JSON",
        "Terminal/Manual",
        "Não",
        "Prototipagem"
    ],
    "MLflow": [
        "mlflow ui",
        "Média",
        "SQL/Filesystem",
        "Web UI",
        "Sim",
        "Produção"
    ]
})

print(comparison.to_string(index=False))

## Resumo e Próximos Passos

### ✅ O que foi coberto

1. **Ultralytics YOLO**
   - Carregamento de modelos pré-treinados
   - Inferência em imagens (local e remota)
   - Estrutura de datasets (images + labels em formato normalizado)

2. **Treinamento e Exportação**
   - Pipeline de treinamento com COCO128
   - Exportação para múltiplos formatos (ONNX, TensorRT, etc.)
   - Fluxo esperado para DeepStream

3. **Análise de Datasets**
   - Exploração de estrutura e volume
   - Distribuição de classes
   - Identificação de desbalanceamentos

4. **Rastreamento de Experimentos**
   - Sistema manual com JSON
   - Setup inicial para MLflow

### 📋 Próximos passos

#### Dia 1 (Hoje)
- [ ] Executar treinamento curto: `python treinar_coco128.py`
- [ ] Revisar outputs em `/workspaces/yolo-pytorch-studies/runs/`
- [ ] Documentar observações

#### Dias 2-3
- [ ] Investigar datasets de veículos e atributos faciais
- [ ] Validar estrutura e qualidade das anotações
- [ ] Propor taxonomia revisada com justificativas

#### Semana 1
- [ ] Setup de MLflow para rastreamento formal
- [ ] Criar guidelines de anotação
- [ ] Planejar próximas coletas de dados

### 🔗 Referências importantes

| Tópico | Link |
|--------|------|
| Documentação Ultralytics | https://docs.ultralytics.com |
| DeepStream SDK | https://docs.nvidia.com/metropolis/deepstream/dev-guide/ |
| YOLO Format | https://docs.ultralytics.com/datasets/detect/ |
| MLflow | https://mlflow.org/docs/latest/ |
| Datasets públicos | VeRi-776, CompCars, Stanford Cars |

### 📝 Dúvidas e decisões pendentes

- [ ] Qual é o tamanho típico dos crops no servidor gRPC?
- [ ] Qual é o pipeline esperado: YOLO detect → classificação de atributos?
- [ ] Há datasets privados já existentes que precisam ser migrados?
- [ ] Qual é a SLA esperada de latência?
- [ ] MLflow será auto-hospedado ou usar W&B?

# Onboarding IA - Plenus Cloud
## Exploração de Frameworks YOLO, DeepStream e Análise de Datasets

Este notebook documenta o processo onboarding para desenvolvimento com modelos de visão computacional, incluindo:
- ✅ Validação do ambiente Ultralytics
- ✅ Exemplos de inferência e treinamento
- ✅ Exploração e análise de datasets
- ✅ Exportação para produção (TensorRT)
- ✅ Rastreamento de experimentos com MLflow

**Data:** Abril de 2026  
**Objetivo:** Compreender os frameworks e propor organização dos datasets